In [122]:
import os
import networkx as nx
from rdkit import Chem
from karateclub.estimator import Estimator
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from karateclub.utils.treefeatures import WeisfeilerLehmanHashing
import numpy as np
from collections import Counter
from ksvd import ApproximateKSVD

In [123]:
class WL_KSVD(Estimator):
    r""" An implementation of WL_KSVD

    Args:
        wl_iterations (int): Number of Weisfeiler-Lehman iterations. Default is 2.
        attributed (bool): Presence of graph attributes. Default is False.
        dimensions (int): Dimensionality of embedding. Default is 128.
        workers (int): Number of cores. Default is 4.
        down_sampling (float): Down sampling frequency. Default is 0.0001.
        epochs (int): Number of epochs. Default is 10.
        learning_rate (float): HogWild! learning rate. Default is 0.025.
        min_count (int): Minimal count of graph feature occurrences. Default is 5.
        seed (int): Random seed for the model. Default is 42.
        erase_base_features (bool): Erasing the base features. Default is False.

        n_vocab: Number of preliminary vocabulary size.  Default is 1000
        n_atoms: Number of dictionary elements (atoms). Default is 128
        n_non_zero_coefs: Number of nonzero coefficients to target. Default is 10
        max_iter: Maximum number of iterations. Default is 10
        tol: Tolerance for error. Default is 1e-6

    """

    def __init__(
        self,
        wl_iterations: int = 2,
        attributed: bool = False,
        dimensions: int = 128,
        workers: int = 4,
        down_sampling: float = 0.0001,
        epochs: int = 10,
        learning_rate: float = 0.025,
        min_count: int = 5,
        seed: int = 42,
        erase_base_features: bool = False,
        n_vocab: int = 1000,
        n_atoms: int = 128,
        n_non_zero_coefs: int = 10,
        max_iter: int = 10,
        tol: float = 1e-6

    ):
        self.wl_iterations = wl_iterations
        self.attributed = attributed
        self.dimensions = dimensions
        self.workers = workers
        self.down_sampling = down_sampling
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.min_count = min_count
        self.seed = seed
        self.erase_base_features = erase_base_features
        self.n_vocab = n_vocab
        self.n_atoms = n_atoms
        self.n_non_zero_coefs = n_non_zero_coefs
        self.max_iter = max_iter
        self.tol = tol

In [124]:
print('Loading NCI1 dataset')

graphs = []
y = []

filepath = "datasets/NCI_full/1total-connect.sdf"

supplier = Chem.SDMolSupplier(filepath, sanitize=False, removeHs=False)
for mol in supplier:
    if mol is None:
        continue

    G = nx.Graph()

    # Add atoms as nodes
    for atom in mol.GetAtoms():
        G.add_node(
            atom.GetIdx(),
            feature=atom.GetSymbol()   # WL uses node labels
        )

    # Add bonds as edges
    for bond in mol.GetBonds():
        G.add_edge(
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx()
        )

    # Get graph label
    # In NCI1, class label is stored as a molecule property
    label = int(float(mol.GetProp("value")))
    graphs.append(G)


    y.append(label)

print(f"Loaded {len(graphs)} graphs")

Loading NCI1 dataset
Loaded 37349 graphs


In [125]:
wl_ksvd = WL_KSVD()

In [126]:
graphs[0].nodes

NodeView((0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43))

In [127]:
# documents = []
# # TODO: parallel implementation
# for graph in graphs[:1]:
#     g = wl_ksvd._check_graph(graph)
#     for node in g.nodes():
#         print(node)
#         if "feature" not in g.nodes[node]:
#             print(g.nodes[node])
#             print("Missing feature:", node)
#         else:
#             print(g.nodes[node])
#             document = WeisfeilerLehmanHashing(
#                 g, wl_ksvd.wl_iterations, True, True)
#
#     documents.append(document)

In [ ]:
documents = []
# TODO: parallel implementation
for graph in graphs:
    g = wl_ksvd._check_graph(graph)
    document = WeisfeilerLehmanHashing(
        g, wl_ksvd.wl_iterations, wl_ksvd.attributed, wl_ksvd.erase_base_features)

    documents.append(document)

In [129]:
#printing the content of a doc
features = [doc.get_graph_features() for doc in documents]

In [130]:
len(features)

37349

In [131]:
features[0]

['3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '4025f7ec584c225eb214f61e8fa3aaba',
 '4',
 '9ded9482dcd25745e629e5fd3afbca73',
 '7f01cbf69516028cdff263ab4ef5c7ee',
 '4',
 'af67ad2ffa467c8bad3f840d7c715bb0',
 'dfd14c0b867116b2789a3cc87801ee0d',
 '4',
 'af67ad2ffa467c8bad3f840d7c715bb0',
 '4154eddd1c77b6f2435115470383c073',
 '3',
 '83330c87ecc2517466a49ffe3555fd98',
 '26748bda89269e5edc7bcfea13c999ac',
 '4',
 '8f9e9234712a641a944b115e2f8d07f3',
 '5b05083c2746f603211da50b62c0c45a',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '796097a7389b46e8234f8bc85fa962fa',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '0c823a59055acdfdb3984bf1e8af3203',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '0c823a59055acdfdb3984bf1e8af3203',
 '4',
 '8f9e9234712a641a944b115e2f8d07f3',
 'b1fb9cffdbc26abd1ebceb04ce119249',
 '4',
 '9ded9482dcd25745e629e5fd3afbca73',
 'c56ff23927b49a2444f3b2f6608992a6',
 '5',
 '5f72132389c5a8352ab504ead192f1ac',
 'f96a606c096ff671e73ae6288eb3f91c',
 '6',
 '140a8e01d81b435e04faf07c3c650dc3

In [132]:
documents = [
    TaggedDocument(words=doc.get_graph_features(), tags=[str(i)])
    for i, doc in enumerate(documents)
]

In [133]:
documents[0]

TaggedDocument(words=['3', '9dafdb095db58e4fb4a689fb082bb849', '4025f7ec584c225eb214f61e8fa3aaba', '4', '9ded9482dcd25745e629e5fd3afbca73', '7f01cbf69516028cdff263ab4ef5c7ee', '4', 'af67ad2ffa467c8bad3f840d7c715bb0', 'dfd14c0b867116b2789a3cc87801ee0d', '4', 'af67ad2ffa467c8bad3f840d7c715bb0', '4154eddd1c77b6f2435115470383c073', '3', '83330c87ecc2517466a49ffe3555fd98', '26748bda89269e5edc7bcfea13c999ac', '4', '8f9e9234712a641a944b115e2f8d07f3', '5b05083c2746f603211da50b62c0c45a', '3', '9dafdb095db58e4fb4a689fb082bb849', '796097a7389b46e8234f8bc85fa962fa', '3', '9dafdb095db58e4fb4a689fb082bb849', '0c823a59055acdfdb3984bf1e8af3203', '3', '9dafdb095db58e4fb4a689fb082bb849', '0c823a59055acdfdb3984bf1e8af3203', '4', '8f9e9234712a641a944b115e2f8d07f3', 'b1fb9cffdbc26abd1ebceb04ce119249', '4', '9ded9482dcd25745e629e5fd3afbca73', 'c56ff23927b49a2444f3b2f6608992a6', '5', '5f72132389c5a8352ab504ead192f1ac', 'f96a606c096ff671e73ae6288eb3f91c', '6', '140a8e01d81b435e04faf07c3c650dc3', '777c0c5a6713

In [134]:
documents[1].words

['4',
 '8f9e9234712a641a944b115e2f8d07f3',
 'dde06bf811e1fd2c3d8e1815a275f0a2',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '366f2a0afb1375b2676325d70a41f25e',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '366f2a0afb1375b2676325d70a41f25e',
 '3',
 '6c7ad2f62b6c2e570f3f66bf94445d25',
 '6b915dcbe44f202d230867315dade558',
 '4',
 'af67ad2ffa467c8bad3f840d7c715bb0',
 '237a39b652a055110760669a78b3dc6d',
 '4',
 'f4490329afa6428ebf9afd9005f27abf',
 'cd91549c92600d50c643cc771e36d017',
 '5',
 '561b3fb1e310ef0f3775af1a5b41f1e9',
 '7f3a05a5f7b0ce3251debb2d8e43911d',
 '5',
 '9d28bfd9cd3dda75e77c320ee43032a4',
 'f0f460e5c83461e48c988bd1e66e8586',
 '5',
 '101640cf30a7e8aa5631df7b2fe6afba',
 'bce941ffc859757229c84704f77ee958',
 '5',
 '101640cf30a7e8aa5631df7b2fe6afba',
 '70be4ccec31f7d4b02af01f6b48f0665',
 '5',
 'daf0ea2428a35efce0f888355b911b7b',
 '6ae5ddc9e3a3409fdca3fd9dae8578fa',
 '5',
 'daf0ea2428a35efce0f888355b911b7b',
 'd698e57dfee614bec0f8f833359cbf20',
 '5',
 '101640cf30a7e8aa5631df7b2fe6afba

In [135]:
d2v_model = Doc2Vec(vector_size=wl_ksvd.n_vocab, min_count=wl_ksvd.min_count, epochs=wl_ksvd.epochs)

In [136]:
total_words, corpus_count = d2v_model.scan_vocab(
            corpus_iterable=documents, corpus_file=None,
            progress_per=10000, trim_rule=None
        )

In [137]:
total_words

2937384

In [138]:
corpus_count

37349

In [139]:
d2v_model.corpus_count = corpus_count
d2v_model.corpus_total_words = total_words
d2v_model.prepare_vocab(update=False, keep_raw_vocab=True, trim_rule=None)

{'drop_unique': 3415,
 'retain_total': 2930939,
 'downsample_unique': 48,
 'downsample_total': 1153401,
 'num_retained_words': 2593}

In [140]:
d2v_model.raw_vocab.items()

dict_items([('3', 189441), ('9dafdb095db58e4fb4a689fb082bb849', 125097), ('4025f7ec584c225eb214f61e8fa3aaba', 25107), ('4', 457711), ('9ded9482dcd25745e629e5fd3afbca73', 11009), ('7f01cbf69516028cdff263ab4ef5c7ee', 35), ('af67ad2ffa467c8bad3f840d7c715bb0', 84303), ('dfd14c0b867116b2789a3cc87801ee0d', 319), ('4154eddd1c77b6f2435115470383c073', 5507), ('83330c87ecc2517466a49ffe3555fd98', 31623), ('26748bda89269e5edc7bcfea13c999ac', 1071), ('8f9e9234712a641a944b115e2f8d07f3', 21627), ('5b05083c2746f603211da50b62c0c45a', 184), ('796097a7389b46e8234f8bc85fa962fa', 20758), ('0c823a59055acdfdb3984bf1e8af3203', 41467), ('b1fb9cffdbc26abd1ebceb04ce119249', 9059), ('c56ff23927b49a2444f3b2f6608992a6', 71), ('5', 309225), ('5f72132389c5a8352ab504ead192f1ac', 25107), ('f96a606c096ff671e73ae6288eb3f91c', 984), ('6', 22158), ('140a8e01d81b435e04faf07c3c650dc3', 2089), ('777c0c5a6713cf6910de853764efaf39', 86), ('97099e8cf9c585fda04ada5c8f4c1c9b', 7172), ('544bc74cae7ae64243d2e6106b865c2c', 20), ('ff43

In [141]:
sorted_vocab = (sorted(d2v_model.raw_vocab.items(), key=lambda item: item[1], reverse=True))
sorted_vocab

[('4', 457711),
 ('5', 309225),
 ('f4490329afa6428ebf9afd9005f27abf', 205684),
 ('3', 189441),
 ('9dafdb095db58e4fb4a689fb082bb849', 125097),
 ('0cce1d019f4dfe3f7e302db488dffe57', 111735),
 ('af67ad2ffa467c8bad3f840d7c715bb0', 84303),
 ('daf0ea2428a35efce0f888355b911b7b', 76201),
 ('d2f28a9d851e79d522463d5b5186f3f8', 68559),
 ('101640cf30a7e8aa5631df7b2fe6afba', 60884),
 ('0c823a59055acdfdb3984bf1e8af3203', 41467),
 ('7fdb9950685020307dbcc84ed2feae5d', 41467),
 ('b3a2337249583f88e793adf89f051b5c', 40610),
 ('801f34fb20fa19a400c8178a3c9a04e3', 34684),
 ('6c7ad2f62b6c2e570f3f66bf94445d25', 32186),
 ('83330c87ecc2517466a49ffe3555fd98', 31623),
 ('59fffba59cd5808102e1b6dc26f44c3c', 26768),
 ('97b76f1a9373332c0e005a2feabf0f5d', 26659),
 ('7afcb8896ce267d8a87338ff310f2278', 26136),
 ('4025f7ec584c225eb214f61e8fa3aaba', 25107),
 ('5f72132389c5a8352ab504ead192f1ac', 25107),
 ('6', 22158),
 ('8f9e9234712a641a944b115e2f8d07f3', 21627),
 ('ae45ec3a9cb6b654b4cfa4d828e64d35', 21627),
 ('796097a7389

In [142]:
len(sorted_vocab)

6008

In [143]:
total = 0
for vocab in sorted_vocab:
    total += int(vocab[1])
total

2937384

In [144]:
trimmed_vocab = sorted_vocab[0:wl_ksvd.n_vocab]
wl_ksvd.n_vocab = len(trimmed_vocab)

In [145]:
trimmed_vocab

[('4', 457711),
 ('5', 309225),
 ('f4490329afa6428ebf9afd9005f27abf', 205684),
 ('3', 189441),
 ('9dafdb095db58e4fb4a689fb082bb849', 125097),
 ('0cce1d019f4dfe3f7e302db488dffe57', 111735),
 ('af67ad2ffa467c8bad3f840d7c715bb0', 84303),
 ('daf0ea2428a35efce0f888355b911b7b', 76201),
 ('d2f28a9d851e79d522463d5b5186f3f8', 68559),
 ('101640cf30a7e8aa5631df7b2fe6afba', 60884),
 ('0c823a59055acdfdb3984bf1e8af3203', 41467),
 ('7fdb9950685020307dbcc84ed2feae5d', 41467),
 ('b3a2337249583f88e793adf89f051b5c', 40610),
 ('801f34fb20fa19a400c8178a3c9a04e3', 34684),
 ('6c7ad2f62b6c2e570f3f66bf94445d25', 32186),
 ('83330c87ecc2517466a49ffe3555fd98', 31623),
 ('59fffba59cd5808102e1b6dc26f44c3c', 26768),
 ('97b76f1a9373332c0e005a2feabf0f5d', 26659),
 ('7afcb8896ce267d8a87338ff310f2278', 26136),
 ('4025f7ec584c225eb214f61e8fa3aaba', 25107),
 ('5f72132389c5a8352ab504ead192f1ac', 25107),
 ('6', 22158),
 ('8f9e9234712a641a944b115e2f8d07f3', 21627),
 ('ae45ec3a9cb6b654b4cfa4d828e64d35', 21627),
 ('796097a7389

In [146]:
trimmed_total = 0
for vocab in trimmed_vocab:
    trimmed_total += int(vocab[1])
trimmed_total

2911799

In [147]:
(trimmed_total/total)*100

99.12898688084364

In [ ]:
sparse_vector = np.zeros([len(documents), wl_ksvd.n_vocab])

In [ ]:
sparse_vector

In [ ]:
i = 0
for corpus in documents:
    words = corpus.words

    words_count = Counter(corpus.words)
    j = 0
    for atom, _ in trimmed_vocab:
        sparse_vector[i][j] = words_count[atom]
        j = j + 1

    i = i + 1

In [ ]:
sparse_vector

In [ ]:
x = sparse_vector

In [ ]:
aksvd = ApproximateKSVD(n_components=wl_ksvd.dimensions, max_iter=wl_ksvd.max_iter, tol=wl_ksvd.tol,
                 transform_n_nonzero_coefs=wl_ksvd.n_non_zero_coefs)
wl_ksvd._dictionary = aksvd.fit(x).components_

wl_ksvd._embedding = aksvd.transform(x)

wl_ksvd.aksvd = aksvd

In [ ]:
wl_ksvd._dictionary